In [ ]:
# Importar librerias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Cargar dataset
df = pd.read_csv("/content/datos.csv")

In [ ]:
# Definir variables predictoras y target
X = df.drop("Salary", axis=1)
y = df["Salary"]

# Columnas categóricas y numéricas
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

# Preprocesamiento
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

# División train/validación
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)

# Función auxiliar de métricas
def metricas(y_real, y_pred):
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return rmse, r2

Algoritmo de penalización

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet

# Lasso
modelo_lasso = Pipeline([("prep", preprocessor), ("model", Lasso(alpha=0.1))])
modelo_lasso.fit(X_train, y_train)
y_pred_lasso = modelo_lasso.predict(X_valid)
rmse_lasso, r2_lasso = metricas(y_valid, y_pred_lasso)

# Ridge
modelo_ridge = Pipeline([("prep", preprocessor), ("model", Ridge(alpha=1.0))])
modelo_ridge.fit(X_train, y_train)
y_pred_ridge = modelo_ridge.predict(X_valid)
rmse_ridge, r2_ridge = metricas(y_valid, y_pred_ridge)

# ElasticNet
modelo_elastic = Pipeline([("prep", preprocessor), ("model", ElasticNet(alpha=0.1, l1_ratio=0.5))])
modelo_elastic.fit(X_train, y_train)
y_pred_elastic = modelo_elastic.predict(X_valid)
rmse_elastic, r2_elastic = metricas(y_valid, y_pred_elastic)

print("Lasso -> RMSE:", rmse_lasso, "R2:", r2_lasso)
print("Ridge -> RMSE:", rmse_ridge, "R2:", r2_ridge)
print("ElasticNet -> RMSE:", rmse_elastic, "R2:", r2_elastic)

Algoritmo Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modelo_rf = Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(n_estimators=100, random_state=42))])
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_valid)
rmse_rf, r2_rf = metricas(y_valid, y_pred_rf)

print("--- Random Forest ---")
print("RMSE:", rmse_rf, "R2:", r2_rf)

# Importancia de variables
importancias_rf = modelo_rf.named_steps["model"].feature_importances_

Algoritmo XGBoost

In [ ]:
from xgboost import XGBRegressor

modelo_xgb = Pipeline([("prep", preprocessor), ("model", XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))])
modelo_xgb.fit(X_train, y_train)
y_pred_xgb = modelo_xgb.predict(X_valid)
rmse_xgb, r2_xgb = metricas(y_valid, y_pred_xgb)

print("--- XGBoost ---")
print("RMSE:", rmse_xgb, "R2:", r2_xgb)

# Importancia de variables
importancias_xgb = modelo_xgb.named_steps["model"].feature_importances_

Red neunoral simple

In [ ]:
from sklearn.neural_network import MLPRegressor

modelo_nn = Pipeline([("prep", preprocessor), ("model", MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500, random_state=42))])
modelo_nn.fit(X_train, y_train)
y_pred_nn = modelo_nn.predict(X_valid)
rmse_nn, r2_nn = metricas(y_valid, y_pred_nn)

print("--- Red Neuronal ---")
print("RMSE:", rmse_nn, "R2:", r2_nn)

Resumen Tabla comparativa

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["Lasso", "Ridge", "ElasticNet", "Random Forest", "XGBoost", "Red Neuronal"],
    "RMSE Validación": [rmse_lasso, rmse_ridge, rmse_elastic, rmse_rf, rmse_xgb, rmse_nn],
    "R2 Validación": [r2_lasso, r2_ridge, r2_elastic, r2_rf, r2_xgb, r2_nn]
})

resultados.set_index("Modelo", inplace=True)
print(resultados.round(4))

Hiperparametros

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# Definir el pipeline
pipe_rf = Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(random_state=42))])

# Definir la grilla de parámetros
param_grid_rf = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

# GridSearchCV
grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=3, scoring="r2", n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Mejores parámetros RF:", grid_rf.best_params_)
print("Mejor R2 RF:", grid_rf.best_score_)

In [ ]:
from xgboost import XGBRegressor

pipe_xgb = Pipeline([("prep", preprocessor), ("model", XGBRegressor(random_state=42))])

param_grid_xgb = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__max_depth": [3, 5, 7]
}

grid_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=3, scoring="r2", n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print("Mejores parámetros XGB:", grid_xgb.best_params_)
print("Mejor R2 XGB:", grid_xgb.best_score_)

Resultado Final

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["Lasso", "Ridge", "ElasticNet", "Random Forest", "XGBoost", "Red Neuronal"],
    "RMSE Validación": [rmse_lasso, rmse_ridge, rmse_elastic, rmse_rf, rmse_xgb, rmse_nn],
    "R2 Validación": [r2_lasso, r2_ridge, r2_elastic, r2_rf, r2_xgb, r2_nn]
})

resultados.set_index("Modelo", inplace=True)
print(resultados.round(4))